# Download STEAD windows (M ≥ 2.0)

Builds the project's training input from STEAD **without downloading the full 91 GB `waveforms.hdf5`**.

STEAD's SeisBench file stores waveforms as uncompressed, contiguous `(1024, 3, 6000)` float32 blocks ("buckets").
So the byte position of every sample is known from the HDF5 header, and only the needed 8 s windows are fetched
with HTTP range requests (~25 GB of traffic in ~44k requests instead of 91 GB). Disk use stays at a few GB.
Speed is limited by your connection: at 4 MB/s the full run takes about 2 hours; Colab is usually much faster.

Runs unchanged on **Google Colab** and **locally** (macOS / Linux / Windows).

**Output** (in the data dir):

| File | Content |
| --- | --- |
| `stead_windows_m2.h5` | dataset `X`, shape `(N, 3, 800)`, float32, raw counts, components Z, N, E, window `[P − 200, P + 600)` samples, gzip 4, 256-row chunks |
| `stead_windows_m2_meta.csv` | one row per row of `X`, same order |
| `README.md` | provenance: date, source URL, SeisBench version, filters, window, counts, skipped traces, git commit |

**Resumable:** work is saved in shards of 8192 traces. If the session or connection drops, run all cells again
and it continues from the last finished shard. If a full local `waveforms.hdf5` already exists, it is read
locally instead of over the network.

**Colab:** with `USE_DRIVE = True`, outputs and shards go to Google Drive (a few GB, fits the free quota).
The final HDF5 file is built on the local Colab disk and then copied to Drive.

## 1. Settings

In [ ]:
OUT_NAME = "stead_windows_m2"
MIN_MAGNITUDE = 2.0
PRE_SAMPLES = 200    # P − 2 s
POST_SAMPLES = 600   # P + 6 s

# Colab only
USE_DRIVE = True     # True: data dir on Google Drive (persists between sessions); False: /content/data (lost on disconnect)
DRIVE_DATA_SUBDIR = "first-three-seconds/data"   # relative to MyDrive

# Mirrors, tried in order. Both serve the same files.
MIRRORS = [
    "https://hifis-storage.desy.de/Helmholtz/HelmholtzAI/SeisBench/datasets/stead/",
    "https://seisbench.gfz.de/mirror/datasets/stead/",
]

# Transfer
N_WORKERS = 16                    # parallel range requests
SHARD_TRACES = 8192               # traces per resumable shard (~80 MB on disk)
MERGE_GAP_BYTES = 256 * 1024      # neighbouring traces closer than this are fetched in one request
MAX_REQUEST_BYTES = 16 * 1024**2  # upper limit for one merged request

TRIAL_TRACES = None    # e.g. 2000: quick end-to-end trial on the first N traces; output goes to <data dir>/_trial/
KEEP_WORK_DIR = False  # True: keep the shard files after the final files are written

## 2. Environment

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
print(f"Python {sys.version.split()[0]} | Colab: {IN_COLAB}")

REQUIRED_MODULES = {"h5py": "h5py", "fsspec": "fsspec", "aiohttp": "aiohttp", "requests": "requests",
                    "tqdm": "tqdm", "pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in REQUIRED_MODULES.items() if importlib.util.find_spec(mod) is None]
if IN_COLAB:
    # seisbench is not needed for the transfer itself; it is installed so its version can be recorded
    to_install = missing + (["seisbench"] if importlib.util.find_spec("seisbench") is None else [])
    if to_install:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *to_install], check=True)
elif missing:
    raise ImportError(f"Missing packages: {missing}. Run: pip install -r requirements.txt (or pip install {' '.join(missing)})")

## 3. Directories

Data dir priority: `F3S_DATA_DIR` environment variable → Colab (Drive or `/content/data`) → `<repo>/data`.

- `<data dir>/stead/`: `metadata.csv` and the cached HDF5 byte index (`waveforms_index.json`)
- `<data dir>/stead_windows_work/`: resumable shards (deleted at the end unless `KEEP_WORK_DIR`)
- `<data dir>/`: final outputs

In [ ]:
import shutil


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / ".git").exists() or (p / "pyproject.toml").is_file() or (p / "spec.md").is_file():
            return p
    return start


repo_root = find_repo_root(Path.cwd().resolve())

if os.environ.get("F3S_DATA_DIR"):
    data_dir = Path(os.environ["F3S_DATA_DIR"]).expanduser()
elif IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        data_dir = Path("/content/drive/MyDrive") / DRIVE_DATA_SUBDIR
    else:
        data_dir = Path("/content/data")
else:
    data_dir = repo_root / "data"

data_dir = data_dir.resolve()
os.environ["F3S_DATA_DIR"] = str(data_dir)

raw_dir = data_dir / "stead"
out_dir = data_dir / "_trial" if TRIAL_TRACES else data_dir
work_dir = out_dir / "stead_windows_work"
# HDF5 is written on a local disk; network mounts like Google Drive handle its random writes badly
build_dir = Path("/content/f3s_build") if IN_COLAB and USE_DRIVE else out_dir
for d in (raw_dir, out_dir, build_dir):
    d.mkdir(parents=True, exist_ok=True)

print("Data dir: ", data_dir)
print("Raw dir:  ", raw_dir)
print("Work dir: ", work_dir)
print("Output:   ", out_dir)

## 4. Byte-range access

Every read is a `(start, length)` byte range of `waveforms.hdf5`: from a full local copy if one exists,
otherwise over HTTP with retries. A response that is not `206 Partial Content` or has the wrong length is rejected.

In [ ]:
import threading
import time

import requests
from tqdm.auto import tqdm

MAX_RETRIES = 12
_tls = threading.local()


def http() -> requests.Session:
    s = getattr(_tls, "session", None)
    if s is None:
        s = requests.Session()
        s.headers["User-Agent"] = "first-three-seconds/download_stead"
        _tls.session = s
    return s


def remote_size(url: str) -> int:
    r = http().head(url, allow_redirects=True, timeout=30)
    r.raise_for_status()
    return int(r.headers["Content-Length"])


def pick_mirror(file_name: str) -> tuple[str, int]:
    for base in MIRRORS:
        url = base + file_name
        try:
            return url, remote_size(url)
        except requests.RequestException as e:
            print(f"Mirror failed: {url} ({e})")
    raise RuntimeError(f"No mirror available for {file_name}")


def http_range(url: str, start: int, length: int) -> bytes:
    last = None
    for attempt in range(MAX_RETRIES):
        try:
            with http().get(url, headers={"Range": f"bytes={start}-{start + length - 1}"},
                            stream=True, timeout=(30, 120)) as r:
                # Check before reading the body: a 200 here would be the whole 91 GB file
                if r.status_code != 206:
                    raise IOError(f"HTTP {r.status_code}, expected 206")
                data = r.content
            if len(data) != length:
                raise IOError(f"got {len(data)} bytes, expected {length}")
            return data
        except (requests.RequestException, OSError) as e:
            last = e
            time.sleep(min(60, 2 ** attempt))
    raise RuntimeError(f"Range {start}+{length} failed after {MAX_RETRIES} attempts: {last}")


def local_range(path: Path, start: int, length: int) -> bytes:
    f = getattr(_tls, "file", None)
    if f is None:
        f = _tls.file = open(path, "rb")
    f.seek(start)
    data = f.read(length)
    if len(data) != length:
        raise IOError(f"short read at {start}")
    return data


def download_file(url: str, total: int, target: Path) -> Path:
    # Resumable full-file download (used for metadata.csv). Writes <name>.download and renames when complete.
    tmp = target.with_name(target.name + ".download")
    if target.is_file() and target.stat().st_size == total:
        return target
    for attempt in range(1, MAX_RETRIES + 1):
        done = tmp.stat().st_size if tmp.is_file() else 0
        if done > total:
            tmp.unlink()
            done = 0
        if done == total:
            break
        try:
            with http().get(url, headers={"Range": f"bytes={done}-"} if done else {}, stream=True, timeout=60) as r:
                r.raise_for_status()
                if done and r.status_code != 206:
                    done = 0
                with open(tmp, "ab" if done else "wb") as f, tqdm(total=total, initial=done, unit="B", unit_scale=True,
                                                                  unit_divisor=1024, desc=target.name) as bar:
                    for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                        f.write(chunk)
                        bar.update(len(chunk))
        except (requests.RequestException, OSError) as e:
            print(f"Attempt {attempt}/{MAX_RETRIES} interrupted: {e}")
            time.sleep(min(60, 2 ** attempt))
    if not tmp.is_file() or tmp.stat().st_size != total:
        raise RuntimeError(f"{target.name}: download incomplete; run the cell again")
    tmp.replace(target)
    return target


# A full local copy (e.g. from an earlier full download) is used if present and complete
wave_url, wave_size = pick_mirror("waveforms.hdf5")
local_wave = next((p for p in (raw_dir / "waveforms.hdf5", data_dir / "seisbench" / "datasets" / "stead" / "waveforms.hdf5")
                   if p.is_file() and p.stat().st_size == wave_size), None)
if local_wave:
    read_range = lambda start, length: local_range(local_wave, start, length)  # noqa: E731
    print("Waveform source: local file", local_wave)
else:
    read_range = lambda start, length: http_range(wave_url, start, length)  # noqa: E731
    print(f"Waveform source: {wave_url} ({wave_size / 1e9:.1f} GB, read by byte ranges)")

## 5. Metadata

Downloads `metadata.csv` (~0.4 GB) if it is not already in `<data dir>/stead/`.

In [ ]:
import numpy as np
import pandas as pd

meta_url, meta_size = pick_mirror("metadata.csv")
meta_path = raw_dir / "metadata.csv"
if not (meta_path.is_file() and meta_path.stat().st_size == meta_size):
    download_file(meta_url, meta_size, meta_path)

meta = pd.read_csv(meta_path, low_memory=False)
print(f"metadata.csv: {len(meta):,} rows, {meta.shape[1]} columns")

# Output column -> metadata column. Names vary between SeisBench versions, so they are checked, not assumed.
OUT_COLUMNS = {
    "trace_name": "trace_name",
    "source_id": "source_id",
    "source_magnitude": "source_magnitude",
    "source_magnitude_type": "source_magnitude_type",
    "source_depth_km": "source_depth_km",
    "source_distance_km": "source_distance_km",
    "station_network_code": "station_network_code",
    "station_code": "station_code",
    "trace_p_arrival_sample": "trace_p_arrival_sample",
    "stead_original_split": "split",
}
missing_cols = [c for c in [*OUT_COLUMNS.values(), "trace_category"] if c not in meta.columns]
if missing_cols:
    raise KeyError(f"Missing metadata columns: {missing_cols}. Available: {list(meta.columns)}")

## 6. HDF5 byte index

Reads only the HDF5 header (dataset names, byte offsets, shapes) once and caches it as
`waveforms_index.json`. Over the network this takes a few minutes because the headers are spread through the file.

In [ ]:
import json

import h5py

index_path = raw_dir / "waveforms_index.json"
index = json.loads(index_path.read_text(encoding="utf-8")) if index_path.is_file() else None
if index is None or index.get("file_size") != wave_size:
    if local_wave:
        fobj = open(local_wave, "rb")
    else:
        import fsspec

        fobj = fsspec.filesystem("http").open(wave_url, "rb", block_size=2**20, cache_type="bytes")
    with fobj, h5py.File(fobj, "r") as h5:
        fmt = {k: (v.decode() if isinstance(v, bytes) else v.item() if hasattr(v, "item") else v)
               for k, v in ((k, h5["data_format"][k][()]) for k in h5["data_format"])}
        buckets = {}
        for name in tqdm(list(h5["data"].keys()), desc="HDF5 header"):
            ds = h5["data"][name]
            if ds.chunks is not None or ds.compression is not None or ds.id.get_offset() is None:
                raise RuntimeError(f"data/{name} is chunked or compressed; byte-range reading is not possible")
            buckets[name] = {"offset": int(ds.id.get_offset()), "shape": list(ds.shape), "dtype": ds.dtype.str}
    index = {"file_size": wave_size, "data_format": fmt, "buckets": buckets}
    tmp = index_path.with_name(index_path.name + ".tmp")
    tmp.write_text(json.dumps(index), encoding="utf-8")
    tmp.replace(index_path)

fmt = index["data_format"]
print("data_format:", fmt)
print("Buckets:", len(index["buckets"]))
assert fmt["component_order"] == "ZNE", fmt
assert fmt["dimension_order"] == "CW", fmt
assert int(fmt["sampling_rate"]) == 100, fmt
dtypes = {b["dtype"] for b in index["buckets"].values()}
assert dtypes == {"<f4"}, dtypes

## 7. Select traces and windows

Filters (spec section 2): `trace_category == "earthquake_local"`, magnitude present, `source_magnitude ≥ 2.0`.

Window: P is rounded to the nearest sample (`floor(p + 0.5)`), the window is `[P − 200, P + 600)`, so P is
always at index 200. Traces whose window does not fit in the 60 s record are skipped and listed in the README.

In [ ]:
W = PRE_SAMPLES + POST_SAMPLES
skipped = {}  # reason -> number of traces

sel = meta[meta["trace_category"] == "earthquake_local"]
n_local = len(sel)
sel = sel[sel["source_magnitude"].notna()]
n_with_mag = len(sel)
sel = sel[sel["source_magnitude"] >= MIN_MAGNITUDE].copy()
n_m2 = len(sel)


def drop(mask: pd.Series, reason: str) -> None:
    global sel
    if mask.any():
        skipped[reason] = int(mask.sum())
        sel = sel[~mask]


drop(sel["trace_p_arrival_sample"].isna(), "P arrival missing")

loc = sel["trace_name"].str.extract(r"^(bucket\d+)\$(\d+),")
sel["bucket"] = loc[0]
sel["row"] = pd.to_numeric(loc[1], errors="coerce")
known = sel["bucket"].isin(index["buckets"].keys()) & sel["row"].notna()
drop(~known, "trace_name not found in waveforms.hdf5")

shapes = sel["bucket"].map(lambda b: index["buckets"][b]["shape"])
sel["n_rows"] = shapes.str[0].astype(np.int64)
sel["n_comp"] = shapes.str[1].astype(np.int64)
sel["n_samples"] = shapes.str[2].astype(np.int64)
sel["row"] = sel["row"].astype(np.int64)
drop(sel["row"] >= sel["n_rows"], "trace_name row outside its bucket")

sel["p_index"] = np.floor(sel["trace_p_arrival_sample"].to_numpy() + 0.5).astype(np.int64)
sel["win_start"] = sel["p_index"] - PRE_SAMPLES
drop(sel["win_start"] < 0, f"P earlier than sample {PRE_SAMPLES} (window starts before the record)")
drop(sel["win_start"] + W > sel["n_samples"], "window ends after the record")
assert (sel["n_comp"] == 3).all()

# Byte span of one trace's window: from component Z's window start to component E's window end
ITEM = 4
offset = sel["bucket"].map(lambda b: index["buckets"][b]["offset"]).astype(np.int64)
row_bytes = sel["n_comp"] * sel["n_samples"] * ITEM
sel["span_start"] = offset + sel["row"] * row_bytes + sel["win_start"] * ITEM
sel["span_len"] = ((sel["n_comp"] - 1) * sel["n_samples"] + W) * ITEM

# File order: reading front to back keeps requests sequential and mergeable
cand = sel.sort_values("span_start", kind="stable").reset_index(drop=True)
if TRIAL_TRACES:
    cand = cand.iloc[:TRIAL_TRACES].reset_index(drop=True)

print(f"earthquake_local:          {n_local:,}")
print(f"  with magnitude:          {n_with_mag:,}")
print(f"  with M >= {MIN_MAGNITUDE}:           {n_m2:,}")
for reason, n in skipped.items():
    print(f"  skipped, {reason}: {n:,}")
print(f"Traces to extract:         {len(cand):,} ({cand['source_id'].nunique():,} events)")

## 8. Plan

Groups neighbouring traces into merged requests and checks disk space.
The shard layout is fingerprinted, so a resumed run never mixes shards from different settings.

In [ ]:
import hashlib


def plan_requests(part: pd.DataFrame) -> list[tuple[int, int, list[tuple[int, int]]]]:
    # -> [(request_start, request_length, [(row_in_part, offset_in_request), ...]), ...]
    reqs = []
    starts = part["span_start"].to_numpy()
    ends = starts + part["span_len"].to_numpy()
    cur = None
    for i, (s, e) in enumerate(zip(starts, ends)):
        if cur and s - cur[1] <= MERGE_GAP_BYTES and e - cur[0] <= MAX_REQUEST_BYTES:
            cur[1] = max(cur[1], e)
            cur[2].append((i, int(s - cur[0])))
        else:
            cur = [int(s), int(e), [(i, 0)]]
            reqs.append(cur)
    return [(s, e - s, members) for s, e, members in reqs]


h5_name, csv_name = f"{OUT_NAME}.h5", f"{OUT_NAME}_meta.csv"
already_built = all((out_dir / n).is_file() for n in (h5_name, csv_name, "README.md")) and not work_dir.is_dir()
if already_built:
    print(f"Outputs already exist in {out_dir}; extraction and writing are skipped. Delete them to rebuild.")

n_shards = -(-len(cand) // SHARD_TRACES)
shards = [cand.iloc[k * SHARD_TRACES:(k + 1) * SHARD_TRACES] for k in range(n_shards)]
plans = [plan_requests(part) for part in shards]
n_requests = sum(len(p) for p in plans)
transfer = sum(length for p in plans for _, length, _ in p)
x_bytes = len(cand) * 3 * W * 4
print(f"Shards: {n_shards} | requests: {n_requests:,} | transfer: {transfer / 1e9:.1f} GB "
      f"(full file {wave_size / 1e9:.1f} GB) | X uncompressed: {x_bytes / 1e9:.2f} GB")

fingerprint = hashlib.sha256(
    json.dumps([wave_size, PRE_SAMPLES, POST_SAMPLES, SHARD_TRACES, MIN_MAGNITUDE]).encode()
    + "\n".join(cand["trace_name"] + "|" + cand["win_start"].astype(str)).encode()
).hexdigest()
manifest_path = work_dir / "manifest.json"
if already_built:
    manifest = {}
elif manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest["fingerprint"] != fingerprint:
        raise RuntimeError(f"{work_dir} holds shards from different settings or metadata. Delete it to start over.")
else:
    work_dir.mkdir(parents=True, exist_ok=True)
    manifest = {"fingerprint": fingerprint, "started_utc": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
                "source_url": wave_url if not local_wave else str(local_wave)}
    manifest_path.write_text(json.dumps(manifest, indent=1), encoding="utf-8")

done_shards = set(range(n_shards)) if already_built else {k for k in range(n_shards) if (work_dir / f"shard_{k:05d}.npz").is_file()}
print(f"Shards already done: {len(done_shards)}/{n_shards}")

# Shards (uncompressed) + final gzip file + its temporary copy in the build dir
need_work = x_bytes * (n_shards - len(done_shards)) / max(n_shards, 1) * 1.05
need_build = x_bytes * 1.05
for path, need in [] if already_built else [(work_dir, need_work), (build_dir, need_build)]:
    free = shutil.disk_usage(path).free
    print(f"{path}: need ~{need / 1e9:.1f} GB, free {free / 1e9:.1f} GB")
    # Google Drive mounts report free space unreliably; the Drive quota is checked at drive.google.com/settings/storage
    if need > free and not (IN_COLAB and USE_DRIVE and str(path).startswith("/content/drive")):
        raise RuntimeError(f"Not enough disk space in {path}")

## 9. Extract

Fetches the windows shard by shard. Each shard is written to a temporary file and renamed when complete,
so an interrupted shard is simply redone. Windows with NaN/Inf values are dropped and counted.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def process_shard(k: int, bar) -> None:
    path = work_dir / f"shard_{k:05d}.npz"
    part = shards[k]
    n_samples = part["n_samples"].to_numpy()
    span_count = part["span_len"].to_numpy() // ITEM
    X = np.empty((len(part), 3, W), dtype=np.float32)

    def fetch(req):
        start, length, members = req
        buf = read_range(start, length)
        for i, rel in members:
            a = np.frombuffer(buf, dtype="<f4", count=span_count[i], offset=rel)
            ns = n_samples[i]
            for c in range(3):
                X[i, c] = a[c * ns:c * ns + W]
        return length, len(members)

    with ThreadPoolExecutor(N_WORKERS) as ex:
        for fut in as_completed([ex.submit(fetch, r) for r in plans[k]]):
            nbytes, ntraces = fut.result()
            bar.update(ntraces)
            bar.set_postfix_str(f"shard {k + 1}/{n_shards}, {nbytes / 1e6:.1f} MB")

    finite = np.isfinite(X).all(axis=(1, 2))
    pos = np.arange(k * SHARD_TRACES, k * SHARD_TRACES + len(part))
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "wb") as f:
        np.savez(f, X=X[finite], pos=pos[finite], dropped=pos[~finite])
    tmp.replace(path)


todo = [k for k in range(n_shards) if k not in done_shards]
done_traces = sum(len(shards[k]) for k in done_shards)
t0 = time.time()
with tqdm(total=len(cand), initial=done_traces, unit="trace", desc="Extract") as bar:
    for k in todo:
        process_shard(k, bar)
print(f"Extraction finished in {(time.time() - t0) / 60:.1f} min")

# ## 10. Write outputs
#
# Joins the shards into `X` (gzip 4, 256-row chunks), writes the metadata CSV in the same row order, and the README.

In [ ]:
import datetime as dt
import importlib.metadata


def pkg_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


def git_commit() -> str:
    try:
        sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo_root, capture_output=True, text=True, check=True).stdout.strip()
        dirty = subprocess.run(["git", "status", "--porcelain"], cwd=repo_root, capture_output=True, text=True).stdout.strip()
        return sha + (" (uncommitted changes)" if dirty else "")
    except (OSError, subprocess.CalledProcessError):
        return "unknown (not run from a git checkout)"


def write_data() -> pd.DataFrame:
    shard_files = [work_dir / f"shard_{k:05d}.npz" for k in range(n_shards)]
    assert all(p.is_file() for p in shard_files), "Some shards are missing; run the Extract cell again"

    pos_parts, dropped = [], []
    for p in shard_files:
        with np.load(p) as z:
            pos_parts.append(z["pos"])
            dropped.extend(z["dropped"].tolist())
    pos = np.concatenate(pos_parts)
    if dropped:
        skipped["NaN/Inf in the window"] = len(dropped)

    h5_tmp = build_dir / (h5_name + ".tmp")
    with h5py.File(h5_tmp, "w") as h5:
        X = h5.create_dataset("X", shape=(len(pos), 3, W), dtype="float32", chunks=(256, 3, W),
                              compression="gzip", compression_opts=4)
        h5.attrs["sampling_rate"] = 100
        h5.attrs["component_order"] = "ZNE"
        h5.attrs["pre_samples"] = PRE_SAMPLES
        h5.attrs["post_samples"] = POST_SAMPLES
        h5.attrs["unit"] = "counts"
        h5.attrs["source"] = "STEAD via SeisBench"
        o = 0
        for p in tqdm(shard_files, desc="Write HDF5"):
            with np.load(p) as z:
                x = z["X"]
            X[o:o + len(x)] = x
            o += len(x)
        assert o == len(pos)

    out_meta = cand.iloc[pos][list(OUT_COLUMNS.values())].set_axis(list(OUT_COLUMNS), axis=1).reset_index(drop=True)
    out_meta.insert(out_meta.columns.get_loc("trace_p_arrival_sample") + 1, "win_p_index", PRE_SAMPLES)
    csv_tmp = out_dir / (csv_name + ".tmp")
    out_meta.to_csv(csv_tmp, index=False, encoding="utf-8", lineterminator="\n")

    shutil.move(str(h5_tmp), str(out_dir / h5_name))
    csv_tmp.replace(out_dir / csv_name)
    print(f"Wrote {out_dir / h5_name} ({(out_dir / h5_name).stat().st_size / 1e9:.2f} GB) and {out_dir / csv_name}")
    return out_meta


def write_readme(out_meta: pd.DataFrame) -> str:
    finished = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
    table = lambda s: "\n".join(f"| {k} | {v:,} |" for k, v in s.items())  # noqa: E731
    title = f"TRIAL RUN ({TRIAL_TRACES} traces): STEAD windows" if TRIAL_TRACES else "STEAD windows"
    readme = f"""# {title}, M ≥ {MIN_MAGNITUDE}

Generated by `notebooks/download_stead.ipynb`. Do not edit by hand.

## Source

- Dataset: STEAD, STanford EArthquake Dataset. Mousavi, S. M., Sheng, Y., Zhu, W., & Beroza, G. C. (2019).
  STanford EArthquake Dataset (STEAD): A Global Data Set of Seismic Signals for AI. IEEE Access, 7, 179464–179476.
  doi:10.1109/ACCESS.2019.2947848
- Waveforms: {manifest['source_url']} (SeisBench format, {wave_size:,} bytes; only the windows were read, by byte ranges)
- Metadata: {meta_url}
- SeisBench version: {pkg_version("seisbench")}
- Download started: {manifest['started_utc']}, finished: {finished} (UTC)
- Git commit: {git_commit()}
- Python {sys.version.split()[0]}, numpy {np.__version__}, pandas {pd.__version__}, h5py {h5py.__version__}

## Filters

1. `trace_category == "earthquake_local"`: {n_local:,} traces
2. `source_magnitude` present: {n_with_mag:,}
3. `source_magnitude >= {MIN_MAGNITUDE}`: {n_m2:,}

## Window

- `[P − {PRE_SAMPLES}, P + {POST_SAMPLES})` samples = P − {PRE_SAMPLES / 100:g} s to P + {POST_SAMPLES / 100:g} s at 100 Hz, {W} samples.
- P = `floor(trace_p_arrival_sample + 0.5)`, so P is always at index {PRE_SAMPLES} (`win_p_index`).
- Raw counts, components Z, N, E. No filtering, detrending or normalization.

## Files

| File | Content |
| --- | --- |
| `{h5_name}` | dataset `X`, shape `({len(out_meta)}, 3, {W})`, float32, gzip 4, chunks `(256, 3, {W})` |
| `{csv_name}` | one row per row of `X`, same order (CSV row i = `X[i]`) |

Rows are in the byte order of the source HDF5, not in `metadata.csv` order. `stead_original_split` is STEAD's own
split, kept for reference only; the project splits by `source_id` (spec section 4).

## Counts

- Traces: {len(out_meta):,}
- Events (`source_id`): {out_meta['source_id'].nunique():,}
- Stations: {len(out_meta[['station_network_code', 'station_code']].drop_duplicates()):,}
- Magnitude: {out_meta['source_magnitude'].min():.2f} to {out_meta['source_magnitude'].max():.2f}

| Magnitude type | Traces |
| --- | --- |
{table(out_meta['source_magnitude_type'].fillna('missing').value_counts())}

| STEAD split | Traces |
| --- | --- |
{table(out_meta['stead_original_split'].value_counts())}

## Skipped traces (after the filters)

| Reason | Traces |
| --- | --- |
{table(skipped) if skipped else "| none | 0 |"}
"""
    (out_dir / "README.md").write_text(readme, encoding="utf-8", newline="\n")
    return readme


if already_built:
    print("Outputs already exist; nothing to write.")
else:
    print(write_readme(write_data()))
    if not KEEP_WORK_DIR:
        shutil.rmtree(work_dir)
        print("Removed", work_dir)

## 11. Verify

Works from the output files only, so it can also be run on its own after cells 1–6.

- File structure and attributes, CSV row count
- No NaN / Inf in `X`
- A few random rows compared against an independent h5py read of the source file
- Plot of 5 random windows with the P arrival marked (spec section 8)

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
h5_name, csv_name = f"{OUT_NAME}.h5", f"{OUT_NAME}_meta.csv"
W = PRE_SAMPLES + POST_SAMPLES
meta_check = pd.read_csv(out_dir / csv_name, low_memory=False)
print(f"{csv_name}: {len(meta_check):,} rows, {meta_check['source_id'].nunique():,} events")
print("Columns:", list(meta_check.columns))

with h5py.File(out_dir / h5_name, "r") as h5:
    X = h5["X"]
    print("X:", X.shape, X.dtype, "chunks", X.chunks, X.compression, X.compression_opts)
    print("attrs:", dict(h5.attrs))
    assert X.shape == (len(meta_check), 3, W) and X.dtype == np.float32
    assert X.chunks == (256, 3, W) and X.compression == "gzip" and X.compression_opts == 4
    assert (meta_check["win_p_index"] == PRE_SAMPLES).all()
    assert meta_check["trace_name"].is_unique

    bad = 0
    for i in tqdm(range(0, len(X), 256 * 64), desc="NaN/Inf check"):
        bad += int((~np.isfinite(X[i:i + 256 * 64])).sum())
    print("Non-finite values:", bad)
    assert bad == 0

    # Independent check: plain h5py slicing of the source file, none of the byte arithmetic above
    idx = np.sort(rng.choice(len(X), size=min(3, len(X)), replace=False))
    if local_wave:
        src = open(local_wave, "rb")
    else:
        import fsspec

        src = fsspec.filesystem("http").open(wave_url, "rb", block_size=2**20, cache_type="bytes")
    with src, h5py.File(src, "r") as wf:
        for i in idx:
            r = meta_check.iloc[i]
            bucket, row = r["trace_name"].split("$")[0], int(r["trace_name"].split("$")[1].split(",")[0])
            s = int(np.floor(r["trace_p_arrival_sample"] + 0.5)) - PRE_SAMPLES
            ref = wf["data"][bucket][row, :, s:s + W]
            assert np.array_equal(ref, X[i]), f"X[{i}] differs from the source"
            print(f"X[{i}] = {r['trace_name']}: matches source")

    show = np.sort(rng.choice(len(X), size=min(5, len(X)), replace=False))
    windows = X[show]

t = (np.arange(W) - PRE_SAMPLES) / 100
fig, axes = plt.subplots(len(show), 1, figsize=(10, 2 * len(show)), sharex=True, squeeze=False)
for ax, i, x in zip(axes[:, 0], show, windows):
    for c, name in enumerate("ZNE"):
        ax.plot(t, x[c], lw=0.6, label=name)
    ax.axvline(0, color="k", ls="--", lw=0.8)
    r = meta_check.iloc[i]
    ax.set_title(f"X[{i}] {r['trace_name']}  M{r['source_magnitude']:.1f} {r['source_magnitude_type']}  "
                 f"{r['source_distance_km']:.0f} km", fontsize=9)
    ax.set_ylabel("counts")
axes[0, 0].legend(loc="upper right", fontsize=8)
axes[-1, 0].set_xlabel("time relative to P (s)")
plt.tight_layout()
plt.show()
print("All checks passed.")